# 批量 TSFM 误差分析与导出

该 notebook 用于遍历全部 TSFM，分别输出：
- 特征桶组合误差统计表（CSV）
- 误差桶特征统计表（CSV）

In [ ]:
import os
from pathlib import Path

from analysis.tsfm_error_analysis import (
    list_tsfms,
    load_tsfm_window_dataframe,
    add_quantile_buckets,
    compute_feature_combo_error_stats,
    build_error_interval_feature_stats,
)

In [ ]:
OUTPUT_ROOT = 'correction_datasets_double_res_0'
REPORT_ROOT = Path('analysis_reports')
REPORT_ROOT.mkdir(exist_ok=True)

feature_cols = [
    'hist_std', 'trend_slope', 'trend_strength',
    'fft_low_ratio', 'fft_high_ratio', 'fft_dominant_freq'
]
error_metric = 'mae'

tsfs = list_tsfms(OUTPUT_ROOT)
print('TSFM count =', len(tsfs), tsfs)

In [ ]:
for tsfm in tsfs:
    df = load_tsfm_window_dataframe(OUTPUT_ROOT, tsfm)
    if df.empty:
        print(f'[skip] {tsfm}: no data')
        continue

    out_dir = REPORT_ROOT / tsfm
    out_dir.mkdir(parents=True, exist_ok=True)

    df_b = add_quantile_buckets(df, feature_cols=feature_cols, n_bins=5)
    combo_stats = compute_feature_combo_error_stats(
        df_b,
        bucket_features=['trend_strength', 'fft_high_ratio'],
        error_metric=error_metric,
    )
    combo_stats.to_csv(out_dir / 'feature_combo_error_stats.csv', index=False)

    err_feat_stats = build_error_interval_feature_stats(
        df,
        error_metric=error_metric,
        feature_cols=feature_cols,
        n_error_bins=6,
    )
    err_feat_stats.to_csv(out_dir / 'error_bucket_feature_stats.csv', index=False)

    print(f'[ok] {tsfm}: {len(df)} windows -> {out_dir}')